In [ ]:
import json, os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()) if 'tournament_sim' not in os.getcwd() else os.getcwd())
sys.path.insert(0, os.path.join('..', 'Q21G-player-whl') if not os.path.exists('Q21G-player-whl') else 'Q21G-player-whl')

RESULTS_PATH = "tournament_sim/results.json" if os.path.exists("tournament_sim/results.json") else "q21_improvements/tournament_sim/results.json"
SCENARIOS_PATH = RESULTS_PATH.replace("results.json", "scenarios_generated.json")

with open(RESULTS_PATH) as f:
    results = json.load(f)
with open(SCENARIOS_PATH) as f:
    scenarios = json.load(f)
print(f"Loaded {len(results)} results, {len(scenarios)} scenarios")

## Scenario Overview

In [ ]:
col_w = {"id": 4, "book_name": 30, "book_hint": 40, "association_word": 20, "source_pdf": 25}
header = (f"{'ID':<{col_w['id']}} {'Book Name':<{col_w['book_name']}} "
          f"{'Book Hint':<{col_w['book_hint']}} {'Association Word':<{col_w['association_word']}} "
          f"{'Source PDF':<{col_w['source_pdf']}}")
sep = "-" * len(header)
print(header)
print(sep)
for s in scenarios:
    sid        = str(s.get("id", "?"))
    book_name  = str(s.get("book_name", ""))[:col_w["book_name"] - 1]
    book_hint  = str(s.get("book_hint", ""))[:col_w["book_hint"] - 1]
    assoc      = str(s.get("association_word", ""))[:col_w["association_word"] - 1]
    src_pdf    = str(s.get("source_pdf", ""))[:col_w["source_pdf"] - 1]
    print(f"{sid:<{col_w['id']}} {book_name:<{col_w['book_name']}} "
          f"{book_hint:<{col_w['book_hint']}} {assoc:<{col_w['association_word']}} "
          f"{src_pdf:<{col_w['source_pdf']}}")
print(f"\nTotal scenarios: {len(scenarios)}")

## Per-Scenario Results

In [ ]:
hdr = f"{'Game':<5} {'Scenario':<30} {'Sent%':>6} {'Word':>5} {'NR':>4} {'Score':>7} {'SDK':>4} {'Spec':>5}"
print(hdr)
print("-" * len(hdr))

for r in results:
    if "error" in r:
        print(f"{str(r.get('game', '?')):<5} {str(r.get('scenario_id', '')):<30} ERROR — {r['error'][:50]}")
        continue

    score   = r["score"]
    dual    = r["dual_scoring"]

    # sentence score: prefer breakdown field, fall back to private_score
    breakdown  = score.get("breakdown", {})
    sent_pct   = breakdown.get("opening_sentence_score", r.get("sent_pct", 0.0))

    # word match: breakdown score >= 80 counts as correct
    word_score = breakdown.get("associative_word_score", 0.0)
    word_ok    = r.get("word_correct", word_score >= 80)

    # NR count
    nr         = r.get("nr", "N/A")

    private    = score.get("private_score", 0.0)
    name       = str(r.get("scenario_name", r.get("scenario_id", "?")))[:29]

    print(f"{r.get('game', '?'):<5} {name:<30} "
          f"{sent_pct:>5.1f}% {'YES' if word_ok else 'NO':>5} "
          f"{str(nr):>4} {private:>7.1f} "
          f"{dual.get('sdk', '?'):>4} {dual.get('spec', '?'):>5}")

## Aggregate KPIs

In [ ]:
good = [r for r in results if "error" not in r]
errors = [r for r in results if "error" in r]

if not good:
    print("No successful results to aggregate.")
else:
    private_scores = [r["score"]["private_score"] for r in good]
    avg_score      = sum(private_scores) / len(private_scores)

    sdk_wins  = sum(1 for r in good if r["dual_scoring"]["sdk"]  == 3)   # >= 80
    spec_wins = sum(1 for r in good if r["dual_scoring"]["spec"] == 3)   # >= 85

    sdk_win_rate  = sdk_wins  / len(good) * 100
    spec_win_rate = spec_wins / len(good) * 100

    word_correct  = sum(1 for r in good if r.get("word_correct", False))
    word_accuracy = word_correct / len(good) * 100

    nr_values = [r["nr"] for r in good if isinstance(r.get("nr"), (int, float))]
    avg_nr    = sum(nr_values) / len(nr_values) if nr_values else None

    print("=" * 45)
    print("           AGGREGATE KPIs")
    print("=" * 45)
    print(f"  Games played     : {len(good)}/{len(results)}  ({len(errors)} errors)")
    print(f"  Avg private score: {avg_score:.1f}")
    print(f"  SDK  win rate    : {sdk_win_rate:.1f}%  ({sdk_wins}/{len(good)} games >= 80)")
    print(f"  Spec win rate    : {spec_win_rate:.1f}%  ({spec_wins}/{len(good)} games >= 85)")
    print(f"  Word accuracy    : {word_accuracy:.1f}%  ({word_correct}/{len(good)} correct)")
    if avg_nr is not None:
        print(f"  Avg NR count     : {avg_nr:.1f}  (over {len(nr_values)} games)")
    else:
        print(f"  Avg NR count     : N/A")
    print("=" * 45)

## SDK vs Spec Scoring Comparison

In [ ]:
SDK_THRESHOLDS  = {3: ">=80 (Win)",  2: ">=60 (Med)",  1: ">=40 (Low)",  0: "<40 (Zero)"}
SPEC_THRESHOLDS = {3: ">=85 (Win)",  2: ">=70 (Med)",  1: ">=50 (Low)",  0: "<50 (Zero)"}

good = [r for r in results if "error" not in r]

# Side-by-side table
hdr = f"{'Game':<5} {'Scenario':<28} {'Score':>7}   {'SDK pts':<12} {'Spec pts':<12} {'Changed?':>8}"
print(hdr)
print("-" * len(hdr))

changed_count = 0
sdk_dist  = {0: 0, 1: 0, 2: 0, 3: 0}
spec_dist = {0: 0, 1: 0, 2: 0, 3: 0}

for r in good:
    sdk_pts  = r["dual_scoring"]["sdk"]
    spec_pts = r["dual_scoring"]["spec"]
    changed  = sdk_pts != spec_pts
    if changed:
        changed_count += 1
    sdk_dist[sdk_pts]   += 1
    spec_dist[spec_pts] += 1

    name    = str(r.get("scenario_name", r.get("scenario_id", "?")))[:27]
    private = r["score"]["private_score"]
    flag    = "<-- DIFF" if changed else ""
    print(f"{r.get('game', '?'):<5} {name:<28} {private:>7.1f}   "
          f"{SDK_THRESHOLDS[sdk_pts]:<12} {SPEC_THRESHOLDS[spec_pts]:<12} {flag:>8}")

print()
print(f"Scenarios where SDK != Spec: {changed_count}/{len(good)}")
print()

# Distribution summary
print(f"{'Points':<8} {'SDK count':>10} {'Spec count':>11}")
print("-" * 32)
for pts in [3, 2, 1, 0]:
    label = f"{pts} pts"
    print(f"{label:<8} {sdk_dist[pts]:>10} {spec_dist[pts]:>11}")

print()
total_sdk  = sum(k * v for k, v in sdk_dist.items())
total_spec = sum(k * v for k, v in spec_dist.items())
print(f"Total league points  — SDK: {total_sdk}  |  Spec: {total_spec}")
if total_sdk:
    delta_pct = (total_spec - total_sdk) / total_sdk * 100
    print(f"Spec vs SDK delta: {delta_pct:+.1f}%")

## Expert Analysis (LLM Reflector)

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
from openai_client import generate

reflector_prompt = f"""You are a Q21 League competition strategy advisor analyzing tournament simulation results.

Results JSON:
{json.dumps(results, indent=2, ensure_ascii=False)[:8000]}

Scenarios:
{json.dumps(scenarios, indent=2, ensure_ascii=False)[:4000]}

Produce expert analysis with these sections:
1. Executive Summary (3-5 sentences)
2. Failure Mode Classification — for each scenario, classify as: RAG Miss, Filter Kill, Council Misjudge, Extraction Error, Word Miss, or Success
3. Pattern Analysis — correlations between hint type, source PDF, and success
4. Scoring Threshold Impact — how many results change between SDK (80/60/40) and spec (85/70/50)
5. Top 3 Actionable Recommendations — ranked by expected impact, with specific code changes

Be specific, not generic. Reference actual scenario names and scores."""

analysis = generate(reflector_prompt, temperature=0.0)
from IPython.display import Markdown, display
display(Markdown(analysis))